# RAG Evaluation Lab

## 0. Overview

이 notebook은 제조 RAG/Agent 답변 품질을 golden dataset, LangSmith tracing, Ragas, custom LLM judge, deterministic check로 평가하기 위한 초기 실험 환경이다.

- `golden dataset`: 시험지/정답지 역할을 한다. 질문, reference answer, 기대 source id, category, difficulty를 담는다.
- `LangSmith`: 실행 trace, route, metadata, tag를 남겨 실험 비교와 원인 분석에 사용한다.
- `Ragas`: RAG 답변의 faithfulness, relevancy, context precision/recall 같은 metric을 계산한다.
- `custom LLM judge`: 제조/안전 답변 기준을 사람이 만든 rubric으로 평가한다.
- `deterministic check`: LLM 없이 검증 가능한 retrieval/citation/error/empty-response 규칙을 확인한다.

초기 단계에서는 일부러 모든 평가 코드를 이 notebook 하나에 둔다. 기준이 안정되면 함수들을 `eval/run_eval.py`, `eval/ragas_eval.py`, pytest checks, LangSmith dataset/experiment로 분리할 수 있다.

## 1. Environment Setup

이 notebook은 `ai_server` 디렉터리에서 실행하는 것을 기본으로 한다. kernel은 `ai_server/.venv`를 사용하고, project import가 실패하지 않도록 `ai_server`를 `sys.path`에 추가한다. repo root 또는 `ai_server/notebooks`에서 실행해도 경로를 자동으로 보정한다.

`.env` 값은 출력하지 않고 존재 여부만 확인한다. `LANGSMITH_TRACING=true`와 `LANGSMITH_PROJECT`가 설정되면 graph invoke config의 tags/metadata가 LangSmith trace에 붙는다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys

try:
    from dotenv import load_dotenv
except Exception as exc:
    raise RuntimeError('python-dotenv is required. Install requirements.txt in ai_server first.') from exc

CWD = Path.cwd().resolve()
if (CWD / 'app').exists() and (CWD / 'requirements.txt').exists():
    AI_SERVER_DIR = CWD
    PROJECT_ROOT = AI_SERVER_DIR.parent
elif (CWD / 'ai_server' / 'app').exists():
    PROJECT_ROOT = CWD
    AI_SERVER_DIR = PROJECT_ROOT / 'ai_server'
elif (CWD.parent / 'app').exists() and (CWD.parent / 'requirements.txt').exists():
    AI_SERVER_DIR = CWD.parent
    PROJECT_ROOT = AI_SERVER_DIR.parent
else:
    raise RuntimeError('Run this notebook from ai_server, repo root, or ai_server/notebooks so app/ is available.')
EVAL_DIR = AI_SERVER_DIR / 'eval'
RESULTS_DIR = EVAL_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ai_server/.env is the single backend/notebook env file.
ENV_PATH = AI_SERVER_DIR / '.env'
if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=False)

EVAL_EXPERIMENT_ID = os.environ.setdefault(
    'EVAL_EXPERIMENT_ID',
    datetime.now(timezone.utc).strftime('eval_%Y%m%d_%H%M%S'),
)

# Keep notebook eval runs out of the default app history unless the caller already set paths.
os.environ.setdefault('HISTORY_DB_PATH', str(RESULTS_DIR / 'agent_runs_eval.sqlite3'))
os.environ.setdefault('LANGGRAPH_CHECKPOINT_DB', str(RESULTS_DIR / 'langgraph_checkpoints_eval.sqlite3'))
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'manufacturing-agent-eval')
if os.environ.get('LANGSMITH_TRACING', '').strip().lower() in {'1', 'true', 'yes', 'y'}:
    os.environ.setdefault('LANGCHAIN_TRACING_V2', 'true')

def env_present(key: str) -> str:
    return 'set' if os.environ.get(key) else 'missing'

required_env = ['LANGSMITH_TRACING', 'LANGSMITH_PROJECT', 'OPENAI_API_KEY']
if os.environ.get('LANGSMITH_TRACING', '').strip().lower() in {'1', 'true', 'yes', 'y'}:
    required_env.append('LANGSMITH_API_KEY')

print(f'Notebook cwd: {CWD}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'AI_SERVER_DIR: {AI_SERVER_DIR}')
print(f'ENV_PATH: {ENV_PATH}')
print(f'EVAL_EXPERIMENT_ID: {EVAL_EXPERIMENT_ID}')
for key in required_env:
    print(f'{key}: {env_present(key)}')

missing = [key for key in required_env if not os.environ.get(key)]
if missing:
    print('Missing env vars:', ', '.join(missing))
    print('Set them in .env or the shell before running paid/remote evaluation cells.')


## 2. Imports

현재 프로젝트 import가 실패하면 어떤 import가 실패했는지 명확히 표시한다.

In [ ]:
import importlib
import json
import re
import time
from typing import Any
from uuid import uuid4

import pandas as pd

if str(AI_SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVER_DIR))
print(f'Added ai_server to sys.path: {AI_SERVER_DIR}')

PROJECT_IMPORT_ERROR = None
try:
    from app.main import root_graph, user_service
    from app.schemas.agent import AgentSendRequest
    from app.services.llm_service import LLMService
except Exception as exc:
    PROJECT_IMPORT_ERROR = exc
    print(f'Project import failed: {type(exc).__name__}: {exc}')

try:
    from datasets import Dataset
except Exception as exc:
    Dataset = None
    print(f'datasets import skipped: {type(exc).__name__}: {exc}')

try:
    import ragas
except Exception as exc:
    ragas = None
    print(f'ragas import skipped: {type(exc).__name__}: {exc}')

print('Imports ready')

## 3. Load Golden Dataset

`id`, `user_input`, `reference`는 필수다. 나머지 필드는 없으면 기본값을 채운다.

In [ ]:
GOLDEN_PATH = EVAL_DIR / 'golden_rag_cases.jsonl'

VALID_POLICIES = {'required', 'optional', 'forbidden'}

def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSONL at {path}:{line_no}: {exc}') from exc
    return rows

def as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []
    return [value]

def normalize_policy(value: Any, *, default: str = 'optional', field: str = 'policy') -> str:
    policy = str(value or default).strip().lower()
    if policy not in VALID_POLICIES:
        raise ValueError(f'{field} must be one of {sorted(VALID_POLICIES)}, got {policy!r}')
    return policy

def validate_golden_cases(cases: list[dict[str, Any]]) -> list[dict[str, Any]]:
    required = {'id', 'user_input', 'reference'}
    normalized: list[dict[str, Any]] = []
    seen_ids: set[str] = set()
    for index, case in enumerate(cases, start=1):
        missing = [field for field in required if not str(case.get(field, '')).strip()]
        if missing:
            raise ValueError(f'Case #{index} is missing required fields: {missing}')
        case_id = str(case['id'])
        if case_id in seen_ids:
            raise ValueError(f'Duplicate case id: {case_id}')
        seen_ids.add(case_id)

        item = dict(case)
        item['expected_source_ids'] = [str(x) for x in as_list(item.get('expected_source_ids'))]
        item['expected_doc_ids'] = [str(x) for x in as_list(item.get('expected_doc_ids'))]
        item['expected_keywords'] = [str(x) for x in as_list(item.get('expected_keywords'))]
        item['forbidden_keywords'] = [str(x) for x in as_list(item.get('forbidden_keywords'))]
        item['expected_path'] = str(item.get('expected_path') or '')
        item['expected_intent'] = str(item.get('expected_intent') or '')
        item['retrieval_policy'] = normalize_policy(
            item.get('retrieval_policy') or ('required' if item.get('requires_retrieval') else 'optional'),
            field='retrieval_policy',
        )
        item['citation_policy'] = normalize_policy(
            item.get('citation_policy') or ('required' if item.get('requires_citation') else 'optional'),
            field='citation_policy',
        )
        item['requires_retrieval'] = item['retrieval_policy'] == 'required'
        item['requires_citation'] = item['citation_policy'] == 'required'
        item['category'] = item.get('category') or 'unknown'
        item['difficulty'] = item.get('difficulty') or 'unknown'
        item['expected_behavior'] = item.get('expected_behavior') or ''
        item['notes'] = item.get('notes') or ''
        normalized.append(item)
    return normalized

golden_cases = validate_golden_cases(load_jsonl(GOLDEN_PATH))
golden_df = pd.DataFrame(golden_cases)
golden_display_columns = [
    'id', 'category', 'difficulty', 'retrieval_policy', 'citation_policy',
    'expected_path', 'expected_intent', 'expected_source_ids', 'expected_doc_ids',
    'expected_keywords', 'forbidden_keywords',
]
display(golden_df[[col for col in golden_display_columns if col in golden_df.columns]])


## 4. Build or Load RAG Graph

이 repo의 실제 진입점은 `app.main.root_graph`, 즉 `RootManufacturingGraph` 인스턴스다. FastAPI `/agent/send`도 같은 객체의 `run()`을 호출한다. 평가는 public API 응답 스키마를 바꾸지 않고 compiled graph를 직접 invoke해 final state를 읽는다.

In [ ]:
EVAL_USER_ID = 'eval_user'

def ensure_eval_user(user_id: str = EVAL_USER_ID) -> None:
    if PROJECT_IMPORT_ERROR:
        raise RuntimeError(f'Project import failed earlier: {PROJECT_IMPORT_ERROR}')
    try:
        user_service.get(user_id)
    except Exception:
        user_service.create({
            'user_id': user_id,
            'display_name': 'Evaluation User',
            'role': 'eval',
            'department': 'quality',
            'preferred_language': 'ko',
            'report_style': 'standard',
        })

def get_graph_runtime():
    if PROJECT_IMPORT_ERROR:
        raise RuntimeError(f'Project import failed earlier: {PROJECT_IMPORT_ERROR}')
    ensure_eval_user()
    return root_graph

graph_runtime = get_graph_runtime()
graph = graph_runtime.graph
print(type(graph_runtime).__name__)
print(graph)

## 5. Single Case Run

첫 번째 case를 LangSmith tags/metadata가 포함된 config로 실행한다. 실제 API key가 없으면 error가 결과 row에 기록되고 notebook 전체는 중단하지 않는다.

In [ ]:
def build_langsmith_config(case: dict[str, Any], *, user_id: str, session_id: str, thread_id: str) -> dict[str, Any]:
    top_k = int(case.get('top_k') or 5)
    return {
        'configurable': {
            'thread_id': thread_id,
            'user_id': user_id,
            'session_id': session_id,
        },
        'tags': [
            'eval',
            'rag',
            str(case.get('category') or 'unknown'),
            str(case.get('retrieval_policy') or 'optional'),
            EVAL_EXPERIMENT_ID,
        ],
        'metadata': {
            'case_id': case['id'],
            'experiment_id': EVAL_EXPERIMENT_ID,
            'dataset': 'golden_rag_v1',
            'prompt_version': 'v1',
            'retriever_top_k': top_k,
            'chunking_version': 'v1',
            'retrieval_policy': case.get('retrieval_policy') or 'optional',
            'citation_policy': case.get('citation_policy') or 'optional',
            'expected_path': case.get('expected_path') or '',
            'expected_intent': case.get('expected_intent') or '',
            'expected_source_ids': list(case.get('expected_source_ids') or []),
            'expected_doc_ids': list(case.get('expected_doc_ids') or []),
            'run_source': 'ai_server/notebooks/01_rag_eval_lab.ipynb',
        },
    }

def build_initial_agent_state(case: dict[str, Any], *, user_id: str = EVAL_USER_ID) -> tuple[dict[str, Any], dict[str, Any]]:
    session_id = str(case.get('session_id') or f'{EVAL_EXPERIMENT_ID}_{case["id"]}')
    send_request = AgentSendRequest(
        user_id=user_id,
        message=str(case['user_input']),
        session_id=session_id,
        top_k=int(case.get('top_k') or 5),
        mode=str(case.get('mode') or 'auto'),
        llm_model=case.get('llm_model'),
    )
    thread_id = str(case.get('thread_id') or f'eval:{EVAL_EXPERIMENT_ID}:{case["id"]}')
    state = {
        'state_schema_version': 2,
        'run_id': str(uuid4()),
        'user_id': user_id,
        'session_id': session_id,
        'thread_id': thread_id,
        'current_user_message': send_request.message,
        'send_request': send_request.model_dump(),
        'warnings': [],
        'errors': [],
        'usage_records': [],
        'trace': [],
        'replan_count': 0,
    }
    config = build_langsmith_config(case, user_id=user_id, session_id=session_id, thread_id=thread_id)
    return state, config


## 6. Helper: Normalize RAG Result

`normalize_rag_result(case, result, latency_seconds, error=None)`는 final state, `AgentResponse`, dict, Document-like object를 가능한 범위에서 공통 평가 schema로 변환한다. 없는 값은 임의로 만들지 않고 빈 값으로 둔다.

In [ ]:
def to_plain(value: Any) -> Any:
    if hasattr(value, 'model_dump'):
        return to_plain(value.model_dump())
    if isinstance(value, dict):
        return {str(k): to_plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_plain(v) for v in value]
    return value

def first_non_empty(*values: Any) -> Any:
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and value.strip():
            return value
        if isinstance(value, (list, dict)) and value:
            return value
        if not isinstance(value, (str, list, dict)):
            return value
    return None

def dedupe_str(values: list[Any]) -> list[str]:
    return list(dict.fromkeys(str(value) for value in values if str(value).strip()))

def extract_retrieved_docs(result_dict: dict[str, Any], response_dict: dict[str, Any]) -> list[Any]:
    candidates = [
        result_dict.get('retrieved_documents'),
        response_dict.get('retrieved_documents'),
        result_dict.get('retrieved_contexts'),
        response_dict.get('retrieved_contexts'),
        result_dict.get('contexts'),
        response_dict.get('contexts'),
    ]
    for candidate in candidates:
        if isinstance(candidate, list):
            return candidate
    return []

def extract_contexts_and_ids(docs: list[Any]) -> tuple[list[str], list[str], list[str]]:
    contexts: list[str] = []
    chunk_ids: list[str] = []
    doc_ids: list[str] = []
    for doc in docs or []:
        item = to_plain(doc)
        if isinstance(item, str):
            if item.strip():
                contexts.append(item)
            continue
        if isinstance(item, dict):
            metadata = item.get('metadata') if isinstance(item.get('metadata'), dict) else {}
            text = first_non_empty(item.get('text'), item.get('page_content'), item.get('content'))
            chunk_id = first_non_empty(item.get('chunk_id'), metadata.get('chunk_id'), item.get('id'), metadata.get('id'))
            doc_id = first_non_empty(item.get('doc_id'), metadata.get('doc_id'), item.get('document_id'), metadata.get('document_id'))
            if text:
                contexts.append(str(text))
            if chunk_id:
                chunk_ids.append(str(chunk_id))
            if doc_id:
                doc_ids.append(str(doc_id))
            continue
        page_content = getattr(doc, 'page_content', None)
        metadata = getattr(doc, 'metadata', {}) or {}
        if page_content:
            contexts.append(str(page_content))
        if isinstance(metadata, dict):
            if metadata.get('chunk_id'):
                chunk_ids.append(str(metadata['chunk_id']))
            if metadata.get('doc_id'):
                doc_ids.append(str(metadata['doc_id']))
    return contexts, dedupe_str(chunk_ids), dedupe_str(doc_ids)

def extract_contexts_and_chunk_ids(docs: list[Any]) -> tuple[list[str], list[str]]:
    contexts, chunk_ids, _ = extract_contexts_and_ids(docs)
    return contexts, chunk_ids

def extract_citation_ids(citations: list[Any]) -> tuple[list[str], list[str], list[str]]:
    chunk_ids: list[str] = []
    doc_ids: list[str] = []
    labels: list[str] = []
    for citation in citations or []:
        item = to_plain(citation)
        if not isinstance(item, dict):
            continue
        chunk_id = first_non_empty(item.get('chunk_id'), item.get('source_id'), item.get('id'))
        doc_id = first_non_empty(item.get('doc_id'), item.get('document_id'))
        label = first_non_empty(item.get('label'), item.get('display_text'))
        if chunk_id:
            chunk_ids.append(str(chunk_id))
        if doc_id:
            doc_ids.append(str(doc_id))
        if label:
            labels.append(str(label))
    return dedupe_str(chunk_ids), dedupe_str(doc_ids), dedupe_str(labels)

def extract_citation_chunk_ids(citations: list[Any]) -> list[str]:
    chunk_ids, _, _ = extract_citation_ids(citations)
    return chunk_ids

def normalize_rag_result(case: dict[str, Any], result: Any, latency_seconds: float, error: str | None = None) -> dict[str, Any]:
    plain_result = to_plain(result)
    result_dict = plain_result if isinstance(plain_result, dict) else {}
    response_dict = to_plain(result_dict.get('response') or result_dict)
    if not isinstance(response_dict, dict):
        response_dict = {}

    plan = response_dict.get('plan') or result_dict.get('plan') or {}
    plain_plan = to_plain(plan)
    plan = plain_plan if isinstance(plain_plan, dict) else {}
    gateway = result_dict.get('intent_gateway') or {}
    if not isinstance(gateway, dict):
        gateway = {}

    docs = extract_retrieved_docs(result_dict, response_dict)
    contexts, chunk_ids, doc_ids = extract_contexts_and_ids(docs)
    citations = response_dict.get('citations') or result_dict.get('citations') or []
    citations = citations if isinstance(citations, list) else []
    citation_chunk_ids, citation_doc_ids, citation_labels = extract_citation_ids(citations)

    route = response_dict.get('route') or result_dict.get('route') or []
    selected_path = first_non_empty(
        result_dict.get('selected_path'),
        response_dict.get('selected_path'),
        gateway.get('selected_path'),
        route[-1] if isinstance(route, list) and route else None,
    )
    intent = first_non_empty(plan.get('intent'), gateway.get('intent'), gateway.get('turn_type'), response_dict.get('intent'))
    response_text = first_non_empty(
        result_dict.get('final_answer'),
        result_dict.get('answer'),
        response_dict.get('final_answer'),
        response_dict.get('answer'),
        response_dict.get('response'),
    ) or ''

    state_errors = result_dict.get('errors') if isinstance(result_dict.get('errors'), list) else []
    error_text = error or ('; '.join(str(item) for item in state_errors) if state_errors else None)
    return {
        'id': case['id'],
        'user_input': case['user_input'],
        'reference': case.get('reference') or '',
        'response': str(response_text),
        'response_chars': len(str(response_text)),
        'retrieved_contexts': contexts,
        'retrieved_chunk_ids': chunk_ids,
        'retrieved_doc_ids': doc_ids,
        'expected_source_ids': list(case.get('expected_source_ids') or []),
        'expected_doc_ids': list(case.get('expected_doc_ids') or []),
        'expected_path': case.get('expected_path') or '',
        'expected_intent': case.get('expected_intent') or '',
        'retrieval_policy': case.get('retrieval_policy') or 'optional',
        'citation_policy': case.get('citation_policy') or 'optional',
        'requires_retrieval': bool(case.get('requires_retrieval')),
        'requires_citation': bool(case.get('requires_citation')),
        'expected_keywords': list(case.get('expected_keywords') or []),
        'forbidden_keywords': list(case.get('forbidden_keywords') or []),
        'expected_behavior': case.get('expected_behavior') or '',
        'notes': case.get('notes') or '',
        'citations': citations,
        'citation_chunk_ids': citation_chunk_ids,
        'citation_doc_ids': citation_doc_ids,
        'citation_labels': citation_labels,
        'selected_path': str(selected_path or ''),
        'intent': str(intent or ''),
        'category': case.get('category') or 'unknown',
        'difficulty': case.get('difficulty') or 'unknown',
        'latency_seconds': round(float(latency_seconds), 3),
        'error': error_text,
    }

def run_single_case(case: dict[str, Any], *, graph_runtime=None, user_id: str = EVAL_USER_ID) -> dict[str, Any]:
    graph_runtime = graph_runtime or get_graph_runtime()
    ensure_eval_user(user_id)
    state, config = build_initial_agent_state(case, user_id=user_id)
    started = time.perf_counter()
    try:
        final_state = graph_runtime.graph.invoke(state, config=config)
        error = None
    except Exception as exc:
        final_state = {}
        error = f'{type(exc).__name__}: {exc}'
    latency = time.perf_counter() - started
    return normalize_rag_result(case, final_state, latency, error=error)

single_case_result = run_single_case(golden_cases[0], graph_runtime=graph_runtime)
pd.DataFrame([single_case_result])


## 6.1 Sanity Check: `root_graph.run()` vs `root_graph.graph.invoke()`

같은 질문을 public runtime entrypoint인 `root_graph.run()`과 compiled graph direct invoke로 각각 실행해 핵심 평가 필드가 어떻게 보이는지 비교한다. `root_graph.run()` 응답에는 `selected_path`가 public field로 직접 없으므로 trace detail의 `path=...`에서 복구한다.

In [ ]:
def selected_path_from_trace(response_dict: dict[str, Any]) -> str:
    for step in response_dict.get('trace') or []:
        detail = str(step.get('detail') if isinstance(step, dict) else getattr(step, 'detail', ''))
        match = re.search(r'path=([A-Za-z0-9_]+)', detail)
        if match:
            return match.group(1)
    return ''

def extract_sanity_fields(result: Any, *, source: str) -> dict[str, Any]:
    result_dict = to_plain(result) if isinstance(to_plain(result), dict) else {}
    response_dict = to_plain(result_dict.get('response') or result_dict)
    if not isinstance(response_dict, dict):
        response_dict = {}
    docs = extract_retrieved_docs(result_dict, response_dict)
    _, chunk_ids = extract_contexts_and_chunk_ids(docs)
    citations = response_dict.get('citations') or result_dict.get('citations') or []
    selected_path = first_non_empty(
        result_dict.get('selected_path'),
        response_dict.get('selected_path'),
        selected_path_from_trace(response_dict),
    ) or ''
    return {
        'source': source,
        'final_answer': first_non_empty(result_dict.get('answer'), response_dict.get('answer'), response_dict.get('response')) or '',
        'selected_path': selected_path,
        'citations': citations if isinstance(citations, list) else [],
        'retrieved_chunk_ids': chunk_ids,
    }

def run_public_runtime_for_sanity(case: dict[str, Any]) -> dict[str, Any]:
    sanity_case = dict(case)
    sanity_case['session_id'] = f'sanity_run_{case["id"]}_{uuid4().hex[:8]}'
    req = AgentSendRequest(
        user_id=EVAL_USER_ID,
        message=str(sanity_case['user_input']),
        session_id=sanity_case['session_id'],
        top_k=int(sanity_case.get('top_k') or 5),
        mode=str(sanity_case.get('mode') or 'auto'),
        llm_model=sanity_case.get('llm_model'),
    )
    try:
        response = graph_runtime.run(req)
        fields = extract_sanity_fields(response, source='root_graph.run')
        fields['error'] = None
        return fields
    except Exception as exc:
        return {'source': 'root_graph.run', 'final_answer': '', 'selected_path': '', 'citations': [], 'retrieved_chunk_ids': [], 'error': f'{type(exc).__name__}: {exc}'}

def run_direct_invoke_for_sanity(case: dict[str, Any]) -> dict[str, Any]:
    sanity_case = dict(case)
    sanity_case['session_id'] = f'sanity_invoke_{case["id"]}_{uuid4().hex[:8]}'
    sanity_case['id'] = f'{case["id"]}-sanity-invoke'
    state, config = build_initial_agent_state(sanity_case, user_id=EVAL_USER_ID)
    try:
        final_state = graph_runtime.graph.invoke(state, config=config)
        fields = extract_sanity_fields(final_state, source='root_graph.graph.invoke')
        fields['error'] = None
        return fields
    except Exception as exc:
        return {'source': 'root_graph.graph.invoke', 'final_answer': '', 'selected_path': '', 'citations': [], 'retrieved_chunk_ids': [], 'error': f'{type(exc).__name__}: {exc}'}

def compare_runtime_sanity(case: dict[str, Any]) -> pd.DataFrame:
    ensure_eval_user(EVAL_USER_ID)
    public_fields = run_public_runtime_for_sanity(case)
    direct_fields = run_direct_invoke_for_sanity(case)
    rows = []
    for field in ['final_answer', 'selected_path', 'citations', 'retrieved_chunk_ids', 'error']:
        left = public_fields.get(field)
        right = direct_fields.get(field)
        rows.append({
            'field': field,
            'root_graph.run': left,
            'root_graph.graph.invoke': right,
            'match': left == right,
        })
    return pd.DataFrame(rows)

sanity_compare_df = compare_runtime_sanity(golden_cases[0])
display(sanity_compare_df)

## 7. Run All Cases With LangSmith Tracing

각 case는 `thread_id=eval:{case_id}` 형태의 session/thread로 실행하고, 실패해도 전체 루프를 중단하지 않는다.

In [ ]:
def run_all_cases(cases: list[dict[str, Any]], *, graph_runtime=None, user_id: str = EVAL_USER_ID) -> list[dict[str, Any]]:
    graph_runtime = graph_runtime or get_graph_runtime()
    rows: list[dict[str, Any]] = []
    for case in cases:
        row = run_single_case(case, graph_runtime=graph_runtime, user_id=user_id)
        rows.append(row)
        status = 'error' if row.get('error') else 'ok'
        print(f'{case["id"]}: {status}, latency={row["latency_seconds"]}s')
    return rows

raw_outputs = run_all_cases(golden_cases, graph_runtime=graph_runtime)
raw_outputs_df = pd.DataFrame(raw_outputs)
display(raw_outputs_df)

## 8. Save Raw Outputs

정규화된 raw output을 JSONL과 CSV로 저장한다. JSONL은 `ensure_ascii=False`를 사용한다.

In [ ]:
RAW_JSONL_PATH = RESULTS_DIR / 'rag_raw_outputs_v1.jsonl'
RAW_CSV_PATH = RESULTS_DIR / 'rag_raw_outputs_v1.csv'

def save_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')

save_jsonl(raw_outputs, RAW_JSONL_PATH)
raw_outputs_df.to_csv(RAW_CSV_PATH, index=False)
print(RAW_JSONL_PATH)
print(RAW_CSV_PATH)

## 9. Deterministic Checks

LLM judge 없이 규칙으로 확인 가능한 항목을 평가한다. `expected_source_ids` 또는 `citation_chunk_ids`가 비어 있으면 해당 check는 `not_applicable`이다.

In [ ]:
DETERMINISTIC_PATH = RESULTS_DIR / 'deterministic_check_results_v1.csv'

def boolish(value: Any) -> bool | None:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        lowered = value.strip().lower()
        if lowered == 'true':
            return True
        if lowered == 'false':
            return False
        return None
    try:
        if value == True:
            return True
        if value == False:
            return False
    except Exception:
        pass
    return None

def pass_fail_or_na(value: bool | str | None) -> str:
    if value is True:
        return 'pass'
    if value is False:
        return 'fail'
    return 'not_applicable'

def all_keywords_present(text: str, keywords: list[str]) -> bool | str:
    terms = [keyword.strip().lower() for keyword in keywords if keyword.strip()]
    if not terms:
        return 'not_applicable'
    lowered = text.lower()
    return all(term in lowered for term in terms)

def no_forbidden_keywords(text: str, keywords: list[str]) -> bool | str:
    terms = [keyword.strip().lower() for keyword in keywords if keyword.strip()]
    if not terms:
        return 'not_applicable'
    lowered = text.lower()
    return not any(term in lowered for term in terms)

def keyword_coverage(text: str, keywords: list[str]) -> float | None:
    terms = [keyword.strip().lower() for keyword in keywords if keyword.strip()]
    if not terms:
        return None
    lowered = text.lower()
    return round(sum(term in lowered for term in terms) / len(terms), 3)

def overlap_pass(expected: list[str], retrieved: list[str]) -> bool | str:
    expected_set = {str(item) for item in expected if str(item).strip()}
    if not expected_set:
        return 'not_applicable'
    retrieved_set = {str(item) for item in retrieved if str(item).strip()}
    return bool(expected_set.intersection(retrieved_set))

def subset_pass(items: list[str], container: list[str]) -> bool | str:
    item_set = {str(item) for item in items if str(item).strip()}
    if not item_set:
        return 'not_applicable'
    container_set = {str(item) for item in container if str(item).strip()}
    return item_set.issubset(container_set)

def policy_pass(policy: str, has_value: bool) -> bool | str:
    policy = normalize_policy(policy, default='optional')
    if policy == 'required':
        return has_value
    if policy == 'forbidden':
        return not has_value
    return 'not_applicable'

def answer_citation_labels(response: str) -> list[str]:
    labels = re.findall(r'\[([A-Za-z0-9_.:-]+)\]', response or '')
    return dedupe_str(labels)

def build_check_failures(row: dict[str, Any], check_values: dict[str, bool | str | None]) -> list[str]:
    failures: list[str] = []
    for name, value in check_values.items():
        if value is False:
            failures.append(name)
    return failures

def run_deterministic_checks(rows: list[dict[str, Any]]) -> pd.DataFrame:
    checks: list[dict[str, Any]] = []
    for row in rows:
        response = str(row.get('response') or '')
        retrieved_contexts = row.get('retrieved_contexts') or []
        retrieved_chunk_ids = [str(x) for x in (row.get('retrieved_chunk_ids') or [])]
        retrieved_doc_ids = [str(x) for x in (row.get('retrieved_doc_ids') or [])]
        expected_source_ids = [str(x) for x in (row.get('expected_source_ids') or [])]
        expected_doc_ids = [str(x) for x in (row.get('expected_doc_ids') or [])]
        citation_chunk_ids = [str(x) for x in (row.get('citation_chunk_ids') or [])]
        citation_doc_ids = [str(x) for x in (row.get('citation_doc_ids') or [])]
        citation_labels = [str(x) for x in (row.get('citation_labels') or [])]
        retrieval_policy = normalize_policy(row.get('retrieval_policy'), default='optional')
        citation_policy = normalize_policy(row.get('citation_policy'), default='optional')

        retrieved_contexts_not_empty = len(retrieved_contexts) >= 1
        response_not_empty = bool(response.strip())
        no_error = not bool(row.get('error'))
        retrieved_id_pool = retrieved_chunk_ids + retrieved_doc_ids
        expected_id_pool = expected_source_ids + expected_doc_ids

        answer_labels = answer_citation_labels(response)
        check_values: dict[str, bool | str | None] = {
            'retrieval_policy_pass': policy_pass(retrieval_policy, retrieved_contexts_not_empty),
            'citation_policy_pass': policy_pass(citation_policy, bool(citation_chunk_ids or citation_doc_ids)),
            'retriever_recall_pass': overlap_pass(expected_id_pool, retrieved_id_pool),
            'citation_integrity_pass': subset_pass(citation_chunk_ids, retrieved_chunk_ids),
            'citation_doc_integrity_pass': subset_pass(citation_doc_ids, retrieved_doc_ids),
            'answer_citation_labels_valid': subset_pass(answer_labels, citation_labels),
            'selected_path_pass': (str(row.get('selected_path') or '') == str(row.get('expected_path') or '')) if row.get('expected_path') else 'not_applicable',
            'intent_pass': (str(row.get('intent') or '') == str(row.get('expected_intent') or '')) if row.get('expected_intent') else 'not_applicable',
            'expected_keywords_pass': all_keywords_present(response, [str(x) for x in (row.get('expected_keywords') or [])]),
            'forbidden_keywords_pass': no_forbidden_keywords(response, [str(x) for x in (row.get('forbidden_keywords') or [])]),
            'no_error': no_error,
            'response_not_empty': response_not_empty,
        }
        bool_values = [value for value in check_values.values() if isinstance(value, bool)]
        pass_count = sum(value is True for value in bool_values)
        fail_count = sum(value is False for value in bool_values)
        pass_rate = round(pass_count / len(bool_values), 3) if bool_values else None
        failures = build_check_failures(row, check_values)

        checks.append({
            'id': row['id'],
            'retrieval_policy': retrieval_policy,
            'citation_policy': citation_policy,
            'retrieved_contexts_not_empty': retrieved_contexts_not_empty,
            'retrieved_count': len(retrieved_contexts),
            'citation_count': len(citation_chunk_ids or citation_doc_ids),
            'expected_keyword_coverage': keyword_coverage(response, [str(x) for x in (row.get('expected_keywords') or [])]),
            **check_values,
            'deterministic_pass_count': pass_count,
            'deterministic_fail_count': fail_count,
            'deterministic_pass_rate': pass_rate,
            'deterministic_failure_reasons': '; '.join(failures),
        })
    return pd.DataFrame(checks)

deterministic_df = run_deterministic_checks(raw_outputs)
deterministic_df.to_csv(DETERMINISTIC_PATH, index=False)
display(deterministic_df)
print(DETERMINISTIC_PATH)


## 10. Ragas Evaluation

Ragas는 LLM judge 비용이 들 수 있다. `reference`가 비어 있거나 `retrieved_contexts`가 비어 있는 case는 일부 metric에서 제외되거나 실패할 수 있다. Ragas 버전별 column/metric 차이가 있으므로 가능한 metric만 실행한다.

설치가 안 되어 있으면 `cd ai_server && pip install -r requirements.txt`로 설치한다.

In [ ]:
RAGAS_PATH = RESULTS_DIR / 'ragas_scores_v1.csv'
RAGAS_METRIC_COLUMNS = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

def load_ragas_metrics() -> tuple[list[Any], list[str], list[str]]:
    metric_specs = [
        ('faithfulness', 'ragas.metrics', 'faithfulness'),
        ('answer_relevancy', 'ragas.metrics', 'answer_relevancy'),
        ('context_precision', 'ragas.metrics', 'context_precision'),
        ('context_recall', 'ragas.metrics', 'context_recall'),
    ]
    metrics: list[Any] = []
    names: list[str] = []
    failures: list[str] = []
    for name, module_name, attr in metric_specs:
        try:
            module = importlib.import_module(module_name)
            metric = getattr(module, attr)
            metrics.append(metric)
            names.append(name)
        except Exception as exc:
            failures.append(f'{name}: {type(exc).__name__}: {exc}')
    return metrics, names, failures

def trim_ragas_columns(df: pd.DataFrame) -> pd.DataFrame:
    keep = ['id'] + [col for col in RAGAS_METRIC_COLUMNS if col in df.columns] + ['ragas_schema', 'ragas_error']
    keep = [col for col in keep if col in df.columns]
    return df[keep].copy()

def ragas_result_to_df(result: Any, ids: list[str]) -> pd.DataFrame:
    if hasattr(result, 'to_pandas'):
        df = result.to_pandas()
    elif hasattr(result, 'to_dataframe'):
        df = result.to_dataframe()
    elif hasattr(result, 'scores'):
        df = pd.DataFrame(result.scores)
    else:
        df = pd.DataFrame(result)
    if 'id' not in df.columns:
        df.insert(0, 'id', ids[: len(df)])
    return trim_ragas_columns(df)

def run_ragas_eval(raw_df: pd.DataFrame) -> pd.DataFrame:
    columns = ['id', *RAGAS_METRIC_COLUMNS, 'ragas_schema', 'ragas_error']
    if ragas is None or Dataset is None:
        reason = 'ragas or datasets is not installed/importable'
        print(reason)
        return pd.DataFrame(columns=columns)
    try:
        from ragas import evaluate
    except Exception as exc:
        print(f'Ragas evaluate import failed: {type(exc).__name__}: {exc}')
        return pd.DataFrame(columns=columns)

    valid = raw_df[
        raw_df['response'].fillna('').astype(str).str.strip().ne('')
        & raw_df['reference'].fillna('').astype(str).str.strip().ne('')
        & raw_df['retrieved_contexts'].apply(lambda value: isinstance(value, list) and len(value) > 0)
    ].copy()
    skipped = len(raw_df) - len(valid)
    if skipped:
        print(f'Ragas skipping {skipped} rows with empty response/reference/retrieved_contexts.')
    if valid.empty:
        return pd.DataFrame(columns=columns)

    metrics, metric_names, metric_failures = load_ragas_metrics()
    if metric_failures:
        print('Metric import failures:')
        for failure in metric_failures:
            print('-', failure)
    if not metrics:
        print('No Ragas metrics were importable.')
        return pd.DataFrame(columns=columns)

    ids = valid['id'].tolist()
    new_rows = [
        {
            'user_input': row['user_input'],
            'response': row['response'],
            'retrieved_contexts': row['retrieved_contexts'],
            'reference': row['reference'],
        }
        for _, row in valid.iterrows()
    ]
    legacy_rows = [
        {
            'question': row['user_input'],
            'answer': row['response'],
            'contexts': row['retrieved_contexts'],
            'ground_truth': row['reference'],
        }
        for _, row in valid.iterrows()
    ]

    errors: list[str] = []
    for label, dataset_rows in [('current', new_rows), ('legacy', legacy_rows)]:
        try:
            dataset = Dataset.from_list(dataset_rows)
            result = evaluate(dataset, metrics=metrics)
            df = ragas_result_to_df(result, ids)
            df['ragas_schema'] = label
            if 'ragas_error' not in df.columns:
                df['ragas_error'] = None
            return trim_ragas_columns(df)
        except Exception as exc:
            errors.append(f'{label}: {type(exc).__name__}: {exc}')
    print('Ragas evaluation failed:')
    for err in errors:
        print('-', err)
    return pd.DataFrame([{'id': case_id, 'ragas_error': '; '.join(errors)} for case_id in ids], columns=columns)

ragas_df = run_ragas_eval(raw_outputs_df)
ragas_df.to_csv(RAGAS_PATH, index=False)
display(ragas_df)
print(RAGAS_PATH)


## 11. Custom LLM Judge

Ragas와 별개로 제품 기준에 가까운 rubric을 둔다. 비용 제어를 위해 CUSTOM_JUDGE_LIMIT 환경변수를 지원한다.

- 기본값은 3개 case다.
- 전체 실행은 CUSTOM_JUDGE_LIMIT=all로 설정한다.
- 첫 case smoke test는 CUSTOM_JUDGE_SMOKE=true일 때만 별도로 실행한다.

Rubric:
1. 질문에 직접 답했는가?
2. 검색 근거와 reference answer에 기반했는가?
3. 근거 없는 단정을 피했는가?
4. 제조/안전 관련 질문에서 위험한 지시를 하지 않았는가?
5. 답변이 과하게 장황하거나 불필요한 보고서 형식으로 흐르지 않았는가?
6. citation 또는 retrieved context와 모순되지 않는가?


In [ ]:
CUSTOM_JUDGE_PATH = RESULTS_DIR / 'custom_llm_judge_scores_v1.csv'

def parse_custom_judge_limit(value: str | None, total: int, *, default: int = 3) -> int | None:
    if value is None or not str(value).strip():
        return min(default, total)
    lowered = str(value).strip().lower()
    if lowered in {'all', 'none', 'full'}:
        return None
    parsed = int(lowered)
    if parsed < 0:
        raise ValueError('CUSTOM_JUDGE_LIMIT must be >= 0 or "all"')
    return min(parsed, total)

CUSTOM_JUDGE_LIMIT = parse_custom_judge_limit(os.getenv('CUSTOM_JUDGE_LIMIT'), len(raw_outputs_df))
CUSTOM_JUDGE_SMOKE = os.getenv('CUSTOM_JUDGE_SMOKE', '').strip().lower() in {'1', 'true', 'yes', 'y'}
print('CUSTOM_JUDGE_LIMIT:', 'all' if CUSTOM_JUDGE_LIMIT is None else CUSTOM_JUDGE_LIMIT)
print('CUSTOM_JUDGE_SMOKE:', CUSTOM_JUDGE_SMOKE)

JUDGE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'score': {'type': 'integer', 'minimum': 1, 'maximum': 5},
        'pass': {'type': 'boolean'},
        'reason': {'type': 'string'},
        'failure_type': {
            'type': 'string',
            'enum': ['none', 'retrieval_failure', 'hallucination', 'unsafe_answer', 'irrelevant_answer', 'citation_issue', 'too_verbose', 'unknown'],
        },
    },
    'required': ['score', 'pass', 'reason', 'failure_type'],
}

JUDGE_SYSTEM_PROMPT = """You are evaluating a Korean manufacturing RAG assistant.
Score the answer from 1 to 5 using this rubric:
1. It directly answers the user question.
2. It is grounded in the retrieved contexts and reference answer.
3. It avoids unsupported certainty.
4. It does not give unsafe manufacturing or maintenance instructions.
5. It is concise and does not drift into an unnecessary report format.
6. It does not contradict citations or retrieved contexts.
Return only JSON matching the provided schema."""

def run_custom_llm_judge(raw_df: pd.DataFrame, *, limit: int | None = 3, llm_service: Any | None = None) -> pd.DataFrame:
    columns = ['id', 'custom_judge_score', 'custom_judge_pass', 'custom_judge_reason', 'custom_judge_failure_type', 'custom_judge_error']
    llm_service = llm_service or getattr(graph_runtime, 'llm_service', None) or LLMService()
    if not getattr(llm_service, 'enabled', False):
        reason = 'LLM service is not enabled. Set OPENAI_API_KEY and LLM_PROVIDER before running custom judge.'
        print(reason)
        return pd.DataFrame(columns=columns)

    candidate_df = raw_df if limit is None else raw_df.head(limit)
    rows: list[dict[str, Any]] = []
    for _, row in candidate_df.iterrows():
        payload = {
            'id': row['id'],
            'user_input': row['user_input'],
            'reference': row.get('reference') or '',
            'response': row.get('response') or '',
            'retrieved_contexts': (row.get('retrieved_contexts') or [])[:5],
            'retrieved_chunk_ids': row.get('retrieved_chunk_ids') or [],
            'retrieved_doc_ids': row.get('retrieved_doc_ids') or [],
            'citations': row.get('citations') or [],
            'expected_behavior': row.get('expected_behavior') or '',
            'expected_keywords': row.get('expected_keywords') or [],
            'forbidden_keywords': row.get('forbidden_keywords') or [],
        }
        try:
            judged = llm_service.generate_json(
                schema_name='rag_eval_judge',
                schema=JUDGE_SCHEMA,
                system_prompt=JUDGE_SYSTEM_PROMPT,
                payload=payload,
                operation='custom_llm_judge',
            )
            if not judged:
                raise RuntimeError(llm_service.last_error or 'Judge returned empty output')
            rows.append({
                'id': row['id'],
                'custom_judge_score': int(judged.get('score') or 0),
                'custom_judge_pass': bool(judged.get('pass')),
                'custom_judge_reason': str(judged.get('reason') or ''),
                'custom_judge_failure_type': str(judged.get('failure_type') or 'unknown'),
                'custom_judge_error': None,
            })
        except Exception as exc:
            rows.append({
                'id': row['id'],
                'custom_judge_score': None,
                'custom_judge_pass': False,
                'custom_judge_reason': '',
                'custom_judge_failure_type': 'unknown',
                'custom_judge_error': f'{type(exc).__name__}: {exc}',
            })
    return pd.DataFrame(rows, columns=columns)

if CUSTOM_JUDGE_SMOKE:
    single_judge_df = run_custom_llm_judge(raw_outputs_df, limit=1)
    display(single_judge_df)

custom_judge_df = run_custom_llm_judge(raw_outputs_df, limit=CUSTOM_JUDGE_LIMIT)
custom_judge_df.to_csv(CUSTOM_JUDGE_PATH, index=False)
display(custom_judge_df)
print(CUSTOM_JUDGE_PATH)


## 12. Compare Results

raw outputs, deterministic checks, Ragas scores, custom judge scores를 `id` 기준으로 합친다.

In [ ]:
MERGED_PATH = RESULTS_DIR / 'merged_eval_results_v1.csv'

def metric_only(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=['id'])
    keep = ['id'] + [col for col in columns if col in df.columns]
    return df[keep].drop_duplicates(subset=['id']).copy()

def compare_results(raw_df: pd.DataFrame, deterministic_df: pd.DataFrame, ragas_df: pd.DataFrame, custom_judge_df: pd.DataFrame) -> pd.DataFrame:
    deterministic_columns = [
        'retrieval_policy_pass', 'citation_policy_pass', 'retrieved_contexts_not_empty',
        'retrieved_count', 'citation_count', 'retriever_recall_pass',
        'citation_integrity_pass', 'citation_doc_integrity_pass', 'answer_citation_labels_valid',
        'selected_path_pass', 'intent_pass', 'expected_keywords_pass', 'expected_keyword_coverage',
        'forbidden_keywords_pass', 'no_error', 'response_not_empty',
        'deterministic_pass_count', 'deterministic_fail_count', 'deterministic_pass_rate',
        'deterministic_failure_reasons',
    ]
    ragas_columns = [*RAGAS_METRIC_COLUMNS, 'ragas_schema', 'ragas_error']
    custom_columns = ['custom_judge_score', 'custom_judge_pass', 'custom_judge_reason', 'custom_judge_failure_type', 'custom_judge_error']

    merged = raw_df.merge(metric_only(deterministic_df, deterministic_columns), on='id', how='left')
    if ragas_df is not None and not ragas_df.empty:
        merged = merged.merge(metric_only(ragas_df, ragas_columns), on='id', how='left')
    if custom_judge_df is not None and not custom_judge_df.empty:
        merged = merged.merge(metric_only(custom_judge_df, custom_columns), on='id', how='left')
    return merged

merged_df = compare_results(raw_outputs_df, deterministic_df, ragas_df, custom_judge_df)
preferred_columns = [
    'id', 'user_input', 'category', 'difficulty', 'selected_path', 'expected_path', 'selected_path_pass',
    'intent', 'expected_intent', 'intent_pass', 'retrieval_policy', 'citation_policy',
    'retrieved_count', 'citation_count', 'retrieved_chunk_ids', 'retrieved_doc_ids',
    'expected_source_ids', 'expected_doc_ids', 'retrieval_policy_pass', 'citation_policy_pass',
    'retriever_recall_pass', 'citation_integrity_pass', 'answer_citation_labels_valid',
    'expected_keywords_pass', 'expected_keyword_coverage', 'forbidden_keywords_pass',
    'deterministic_pass_rate', 'deterministic_failure_reasons',
    'faithfulness', 'answer_relevancy', 'response_relevancy', 'context_precision', 'context_recall',
    'custom_judge_score', 'custom_judge_pass', 'custom_judge_failure_type', 'latency_seconds', 'error',
]
display_columns = [col for col in preferred_columns if col in merged_df.columns]
merged_df.to_csv(MERGED_PATH, index=False)
display(merged_df[display_columns])
print(MERGED_PATH)


## 13. Failure Analysis

에러, 빈 retrieval, recall/citation 실패, 낮은 Ragas 점수, custom judge fail, 예상과 다른 route 후보를 빠르게 본다.

In [ ]:
FAILURE_PATH = RESULTS_DIR / 'failure_cases_v1.csv'

def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series([False] * len(df), index=df.index)
    return pd.to_numeric(df[column], errors='coerce')

def split_reason_text(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [part.strip() for part in str(value).split(';') if part.strip()]

def row_failure_reasons(row: pd.Series) -> list[str]:
    reasons = split_reason_text(row.get('deterministic_failure_reasons'))
    if str(row.get('error') or '').strip():
        reasons.append('runtime_error')
    if pd.notna(row.get('faithfulness')) and float(row.get('faithfulness')) < 0.6:
        reasons.append('low_faithfulness')
    if pd.notna(row.get('context_recall')) and float(row.get('context_recall')) < 0.6:
        reasons.append('low_context_recall')
    if row.get('custom_judge_pass') is False or str(row.get('custom_judge_pass')).strip().lower() == 'false':
        reasons.append(f'custom_judge_{row.get("custom_judge_failure_type") or "fail"}')
    if not str(row.get('selected_path') or '').strip():
        reasons.append('missing_selected_path')
    return list(dict.fromkeys(reasons))

def build_failure_cases(merged_df: pd.DataFrame) -> pd.DataFrame:
    failures = merged_df.copy()
    failures['failure_reasons'] = failures.apply(lambda row: '; '.join(row_failure_reasons(row)), axis=1)
    failures = failures[failures['failure_reasons'].fillna('').astype(str).str.strip().ne('')].copy()
    return failures

failure_cases_df = build_failure_cases(merged_df)
failure_cases_df.to_csv(FAILURE_PATH, index=False)
failure_display_columns = ['id', 'category', 'selected_path', 'intent', 'failure_reasons', *display_columns]
failure_display_columns = list(dict.fromkeys([col for col in failure_display_columns if col in failure_cases_df.columns]))
display(failure_cases_df[failure_display_columns])
print(FAILURE_PATH)


## 14. Result Visualization

아래 셀은 merged 결과를 category, route, deterministic check, failure reason, Ragas/custom judge metric 기준으로 빠르게 시각화한다. 차트 생성에는 `matplotlib`이 필요하다.


In [ ]:
VISUALIZATION_DIR = RESULTS_DIR / 'figures'
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print(f'matplotlib import skipped: {type(exc).__name__}: {exc}')

def bool_rate(df: pd.DataFrame, column: str) -> float | None:
    if column not in df.columns:
        return None
    values = df[column].map(boolish)
    values = values.dropna()
    if values.empty:
        return None
    return round(float(values.mean()), 3)

def save_bar(series: pd.Series, *, title: str, ylabel: str, filename: str, ylim: tuple[float, float] | None = None) -> None:
    if plt is None or series.empty:
        return
    fig, ax = plt.subplots(figsize=(max(6, min(12, len(series) * 1.2)), 4))
    series.plot(kind='bar', ax=ax, color='#2f6f73')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    if ylim:
        ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(VISUALIZATION_DIR / filename, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

summary_rows: list[dict[str, Any]] = []
for category, group in merged_df.groupby('category', dropna=False):
    summary_rows.append({
        'category': category,
        'cases': len(group),
        'avg_latency_seconds': round(pd.to_numeric(group.get('latency_seconds'), errors='coerce').mean(), 3),
        'retrieval_rate': bool_rate(group, 'retrieved_contexts_not_empty'),
        'retrieval_policy_pass_rate': bool_rate(group, 'retrieval_policy_pass'),
        'citation_policy_pass_rate': bool_rate(group, 'citation_policy_pass'),
        'keyword_pass_rate': bool_rate(group, 'expected_keywords_pass'),
        'path_pass_rate': bool_rate(group, 'selected_path_pass'),
        'intent_pass_rate': bool_rate(group, 'intent_pass'),
        'custom_judge_pass_rate': bool_rate(group, 'custom_judge_pass'),
        'avg_deterministic_pass_rate': round(pd.to_numeric(group.get('deterministic_pass_rate'), errors='coerce').mean(), 3),
    })
summary_df = pd.DataFrame(summary_rows).sort_values(['category']).reset_index(drop=True)
display(summary_df)

check_columns = [
    'retrieval_policy_pass', 'citation_policy_pass', 'retriever_recall_pass',
    'citation_integrity_pass', 'answer_citation_labels_valid', 'selected_path_pass',
    'intent_pass', 'expected_keywords_pass', 'forbidden_keywords_pass',
]
check_rates = {
    column: bool_rate(merged_df, column)
    for column in check_columns
    if bool_rate(merged_df, column) is not None
}
check_rate_series = pd.Series(check_rates).sort_values()
display(check_rate_series.rename('pass_rate').to_frame())
save_bar(check_rate_series, title='Deterministic Check Pass Rate', ylabel='pass rate', filename='deterministic_check_pass_rate.png', ylim=(0, 1))

route_counts = merged_df['selected_path'].fillna('missing').replace('', 'missing').value_counts()
display(route_counts.rename('case_count').to_frame())
save_bar(route_counts, title='Selected Path Distribution', ylabel='case count', filename='selected_path_distribution.png')

reason_counts: dict[str, int] = {}
if 'failure_reasons' not in merged_df.columns and 'failure_cases_df' in globals():
    reason_source = failure_cases_df
else:
    reason_source = merged_df
for reason_text in reason_source.get('failure_reasons', pd.Series(dtype=str)).fillna('').astype(str):
    for reason in split_reason_text(reason_text):
        reason_counts[reason] = reason_counts.get(reason, 0) + 1
reason_series = pd.Series(reason_counts).sort_values(ascending=False)
display(reason_series.rename('case_count').to_frame())
save_bar(reason_series, title='Failure Reason Counts', ylabel='case count', filename='failure_reason_counts.png')

ragas_metric_series = pd.Series({
    metric: pd.to_numeric(merged_df[metric], errors='coerce').mean()
    for metric in RAGAS_METRIC_COLUMNS
    if metric in merged_df.columns
}).dropna().sort_values()
display(ragas_metric_series.rename('average_score').to_frame())
save_bar(ragas_metric_series, title='Average Ragas Scores', ylabel='score', filename='average_ragas_scores.png', ylim=(0, 1))

if 'custom_judge_score' in merged_df.columns:
    custom_score = pd.to_numeric(merged_df['custom_judge_score'], errors='coerce').dropna()
    display(custom_score.describe().rename('custom_judge_score'))
    if plt is not None and not custom_score.empty:
        fig, ax = plt.subplots(figsize=(6, 4))
        custom_score.plot(kind='hist', bins=[1, 2, 3, 4, 5, 6], ax=ax, color='#7a5c1f', rwidth=0.85)
        ax.set_title('Custom Judge Score Distribution')
        ax.set_xlabel('score')
        ax.set_ylabel('case count')
        ax.set_xticks([1, 2, 3, 4, 5])
        ax.grid(axis='y', alpha=0.25)
        fig.tight_layout()
        fig.savefig(VISUALIZATION_DIR / 'custom_judge_score_distribution.png', dpi=160, bbox_inches='tight')
        plt.show()
        plt.close(fig)

latency_series = pd.to_numeric(merged_df.set_index('id')['latency_seconds'], errors='coerce').dropna().sort_values()
display(latency_series.rename('latency_seconds').to_frame())
save_bar(latency_series, title='Latency by Case', ylabel='seconds', filename='latency_by_case.png')

print(f'Figures saved to: {VISUALIZATION_DIR}')


## 15. Human Review Notes

- Codex가 만든 golden dataset reference는 임시 초안이므로 사람이 검수해야 한다.
- `expected_source_ids`와 `expected_doc_ids`는 실제 vector store chunk를 확인한 뒤 계속 업데이트해야 한다.
- `retrieval_policy`와 `citation_policy`는 `required`, `optional`, `forbidden` 중 하나로 둔다.
- Ragas 점수는 절대적인 정답이 아니라 비교/추세 확인용이다.
- LLM judge도 완벽하지 않으므로 사람 평가와 alignment가 필요하다.
- safety policy, LOTO/정비 절차, 위험 지시 제한 문구는 조직 기준과 현장 규정에 맞게 검수해야 한다.
- 평가가 안정되면 나중에 `run_eval.py`, `ragas_eval.py`, pytest로 분리할 수 있다.
- 지금 단계에서는 일부러 notebook 하나에 모아둔다.
